In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
#!pip install langchain==0.1.16
!pip install langchain-openai

In [ ]:
!pip install faiss-gpu-cu12

In [1]:
!pip install -qU flashrank

In [7]:
import bs4
#from langchain import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [8]:
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_path = (
        "https://m.sports.naver.com/wfootball/article/382/0001249371",
        "https://m.sports.naver.com/wfootball/article/413/0000212276",
    ),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class":["_article_content","ArticleHead_article_head_title__YUNFf"]},
        )
    ),
)

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs

문서의 수: 2


[Document(metadata={'source': 'https://m.sports.naver.com/wfootball/article/382/0001249371'}, page_content='ATM 이적과 PSG 재계약 제안이 동시에? 인생의 기로에 선 이강인…‘골든보이’의 미래는 어떻게 열릴까?PSG 이강인이 1월 중요한 선택의 기로에 놓였다. ATM와 강하게 연결되고 있는 가운데 후반기 거취에 시선이 모아진다. 사진출처｜프랑스 리그앙 페이스북PSG 이강인이 1월 중요한 선택의 기로에 놓였다. ATM와 강하게 연결되고 있는 가운데 후반기 거취에 시선이 모아진다. 사진출처｜PSG 페이스북[스포츠동아 남장현 기자] 파리 생제르맹(PSG·프랑스) 이강인(25)이 미래가 걸린 선택의 기로에 섰다. FC바르셀로나, 레알 마드리드와 함께 스페인 프리메라리가 ‘삼대장’으로 불리우는 명문 아틀레티코 마드리드(ATM)의 러브콜을 받으면서다. 이적설은 스페인에서 먼저 시작됐다. 유력 매체 아스(AS)가 지난 주말 “이강인이 ATM의 겨울 이적시장 우선 영입 대상이다. 협상에 속도를 내고 있다”고 최초 보도한 뒤 여러 외신이 관련 내용을 꾸준히 다루고 있다. 또 따른 스페인 유력지 마르카가 “ATM이 이강인 영입을 서두르고 있다. 선수가 PSG를 떠나고 싶어해 영입 가능성이 있다”고 전했고, 유럽 내 이적시장 소식에 강한 프랑스 매체 풋메르카토 역시 20일(한국시간) “ATM이 PSG 핵심 자원으로 분류된 이강인 영입을 원한다”고 밝혔다. 아스에 따르면 마테우 알레마니 ATM 단장이 최근 프랑스 파리를 찾았고, 17일 파르크 데 프랭스서 열린 프랑스 리그앙 PSG-릴전을 관전했다. 허벅지 근육 부상에서 회복 중인 이강인이 출전 명단에 포함되지 않은 경기다. 그 후의 동선은 정확히 파악되지 않았으나 스페인 언론들은 알레마니 단장이 PSG와 접촉해 협상 테이블을 차렸을 것으로 내다본다. ATM의 ‘이강인 사랑’은 꽤 오래됐다. 2023년 여름 마요르카(스페인)를 떠나 PSG로 향

## RecursiveCharacterTextSplitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
splits = text_splitter.split_documents(docs)
len(splits)

6

In [10]:
splits

[Document(metadata={'source': 'https://m.sports.naver.com/wfootball/article/382/0001249371'}, page_content='ATM 이적과 PSG 재계약 제안이 동시에? 인생의 기로에 선 이강인…‘골든보이’의 미래는 어떻게 열릴까?PSG 이강인이 1월 중요한 선택의 기로에 놓였다. ATM와 강하게 연결되고 있는 가운데 후반기 거취에 시선이 모아진다. 사진출처｜프랑스 리그앙 페이스북PSG 이강인이 1월 중요한 선택의 기로에 놓였다. ATM와 강하게 연결되고 있는 가운데 후반기 거취에 시선이 모아진다. 사진출처｜PSG 페이스북[스포츠동아 남장현 기자] 파리 생제르맹(PSG·프랑스) 이강인(25)이 미래가 걸린 선택의 기로에 섰다. FC바르셀로나, 레알 마드리드와 함께 스페인 프리메라리가 ‘삼대장’으로 불리우는 명문 아틀레티코 마드리드(ATM)의 러브콜을 받으면서다. 이적설은 스페인에서 먼저 시작됐다. 유력 매체 아스(AS)가 지난 주말 “이강인이 ATM의 겨울 이적시장 우선 영입 대상이다. 협상에 속도를 내고 있다”고 최초 보도한 뒤 여러 외신이 관련 내용을 꾸준히 다루고 있다. 또 따른 스페인 유력지 마르카가 “ATM이 이강인 영입을 서두르고 있다. 선수가 PSG를 떠나고 싶어해 영입 가능성이 있다”고 전했고, 유럽 내 이적시장 소식에 강한 프랑스 매체 풋메르카토 역시'),
 Document(metadata={'source': 'https://m.sports.naver.com/wfootball/article/382/0001249371'}, page_content='유력지 마르카가 “ATM이 이강인 영입을 서두르고 있다. 선수가 PSG를 떠나고 싶어해 영입 가능성이 있다”고 전했고, 유럽 내 이적시장 소식에 강한 프랑스 매체 풋메르카토 역시 20일(한국시간) “ATM이 PSG 핵심 자원으로 분류된 이강인 영입을 원한다”고 밝혔다. 아스에 따르면 마테우 알레마니 ATM 단장이 최근 프랑스 파리를 찾

## FAISS(Vectorstore)를 활용하여 청크를 바탕으로 문서의 벡터 표현을 생성

In [14]:
import os

# 여기에 실제 OpenAI API 키를 입력하세요
os.environ["OPENAI_API_KEY"] = ""

# 벡터스토어를 생성.
vectorstore = FAISS.from_documents(
    documents=splits, embedding=OpenAIEmbeddings(model="text-embedding-3-small")
)

# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ca234b6b320>, search_kwargs={})

In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#Question:
{question}

#Context:
{context}

#Answer:"""
)

In [30]:
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 체인을 생성합니다.
rag_chain = (
    {
        # itemgetter를 사용하거나 람다 함수를 사용하여 'question' 문자열만 retriever에 전달합니다.
        "context": (lambda x: x["question"]) | retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

## 스트리밍 출력

In [35]:
rag_chain.invoke({"question": "이강인의 현재 상황에 대해 설명해주세요"})


'이강인은 현재 파리 생제르맹(PSG)에서 뛰고 있으며, 그의 미래에 대한 여러 이적설이 존재합니다. 아틀레티코 마드리드(ATM)와의 이적설이 강하게 제기되고 있으며, 이강인은 PSG를 떠나고 싶어하는 것으로 보도되고 있습니다. 그러나 PSG는 이강인을 잔류시키고 재계약 협상에 나설 계획인 것으로 알려졌습니다. 이강인은 현재 부상으로 인해 개인 훈련을 진행 중이며, 아직 팀 훈련에 복귀하지 않은 상태입니다. 이강인의 이적 여부는 아직 확정되지 않았으며, 그의 미래는 여러 선택의 기로에 놓여 있습니다.'

In [36]:
rag_chain.invoke({"question": "이강인은 지금 어디 팀으로부터 이적 오퍼를 받고 있나요?"})


'이강인은 현재 스페인 라리가의 아틀레티코 마드리드로부터 이적 오퍼를 받고 있습니다.'

In [37]:
rag_chain.invoke({"question": "이강인의 별명은 무엇인가요?"})


'주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다.'

In [38]:
rag_chain.invoke({"question": "이강인은 현재 부상상태인가요?"})


'이강인은 현재 부상 상태입니다. 그는 햄스트링 부상으로 인해 몇 주 동안 경기에 출전하지 못하고 있으며, 개인 훈련을 진행하고 있는 것으로 보입니다. 아직 팀 훈련에 복귀하지 못한 상태입니다.'